<a href="https://colab.research.google.com/github/chenharry66/adversial-audio-experiments/blob/main/01_getting_my_footing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from scipy.io import wavfile

rate, data = wavfile.read('/content/drive/MyDrive/FOLDER/01/recording1.wav')

data.dtype


dtype('int16')

In [3]:
import numpy as np

data = data / 32768

In [12]:
data.shape

(612793,)

Pcm amplitude ranges from -32768 to 32,767

In [4]:
from IPython.display import Audio


audio = Audio(data = data, rate = rate, normalize=False)

audio

In [5]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch


processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")


model.eval()

input_values = processor(data, return_tensors = "pt", sampling_rate = rate).input_values

with torch.inference_mode():
    logits = model(input_values).logits

# logits = model.(input_values).logits



Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
logits.shape

predicted_ids = torch.argmax(logits, dim = -1)

transcription = processor.decode(predicted_ids[0])

In [7]:
predicted_ids.shape

# means theres a row axis need to pass in just a list

torch.Size([1, 1914])

In [13]:
transcription = processor.decode(predicted_ids[0])
transcription
reference = "With one reservation: I have a childlike conviction that the sufferings will be healed and smoothed over that the whole offensive comedy of human contradictions will disappear like a pitiful mirage a vile concction of mans Euclidean mind feeble and puny as an atom and that ultimately at the world's finale in the moment of eternal harmony there will occur and be revealed something so precious that it will suffice for all hearts to allay all indignation to redeem all human villainy all bloodshed"""


In [9]:
!pip install werpy

In [14]:
import werpy as wp

wp.wer(wp.normalize(reference), wp.normalize(transcription))

0.17857142857142858

### First mini experiment: add a super basic low-pass filter see if that changes the transcription results at all

In [15]:
from scipy.fft import rfft

def low_pass_filter(wave, threshold):
    coefficients = rfft(wave)
    freqs = coefficients.rfftfreq(n= len(data), d = 1/rate)


    # perform a rfft